# 1 · The design space

In [6]:
# This is just the code header, ignore it.
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import dse
import dse.plots as plots
from dse import DATA, WORK

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)

A matrix–vector product on one rank of **64 DPUs**. Each DPU runs up to **24 tasklets** (hardware threads) and has its own memory (MRAM, 64 MB) and a small scratchpad (WRAM, 64 KB).

In [7]:
mlir = DATA / "small" / "gemv.mlir"
dse.show_mlir(mlir)

The code describes both the platform (`#upmem = #upmem.platform<...>`) and the program (`cinm.op.gemv`).

To run gemv on that platform, we need to 
- Find a suitable DPU and tasklet count
- Tile the loops to create tiles for the MRAM and WRAM memory levels
- Distribute tiles over DPUs/tasklets

These parameters are summarized below:

In [8]:
space = dse.dump_space(mlir) # Ask Cinnamon to create the design space and dump it to json
dse.cinm.params_table(space)

,parameter,kind,values,count,meaning
0,dpus,integer,64 … 64,1,number of DPUs the workgroup spans
1,tasklets,integer,1 … 24,24,tasklets per DPU
2,gemv.M.mram,integer,"1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024",11,mram tile size of iteration dim M
3,gemv.M.wram,integer,1 … 1024,1024,wram tile size of iteration dim M
4,gemv.K.mram,integer,"1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024",11,mram tile size of iteration dim K
5,gemv.K.wram,integer,1 … 1024,1024,wram tile size of iteration dim K
6,gemv.order,permutation,1 … 2,2,which iteration dim occupies which workgroup axis


Let's see how many parameter combinations are in that space:

In [9]:
print(dse.cinm.describe_space(space))

Cartesian product: 6,090,130,000 combinations


That's a lot, but many of them are **infeasible**, in the sense that they couldn't be compiled into a valid program.

For instance, WRAM tiles must be contained in MRAM tiles. That means values of `gemv.M.mram` constrain `gemv.M.wram` (and same for `K`).

Cinnamon can generate these constraints and uses a constraint solver to filter out the infeasible configurations.

In [10]:
print(dse.cinm.describe_space(space, feasible=True))

Cartesian product: 6,090,130,000 combinations
Feasible:          3,234 configurations (5.3e-05 %)
Built in 0.01 s by the constraint solver (6,491 search nodes)


About 1 in 2 million combinations is feasible.

These are the constraints Cinnamon generated and solved:

In [11]:
pd.set_option("display.max_colwidth", 120)
dse.cinm.constraints_table(space)

,constraint,disposition
0,1024 % gemv.M.mram == 0,static-filter
1,1024 % gemv.K.mram == 0,static-filter
2,(gemv.M.wram | gemv.M.mram),solved
3,(gemv.K.wram | gemv.K.mram),solved
4,(gemv.M.mram | 1024),solved
5,(gemv.K.mram | 1024),solved
6,prod(extent / block) == dpus * tasklets,solved
7,tasklets * sum of operand tiles <= mram (assuming no sharing),solved
8,tasklets * sum of operand tiles <= wram (assuming no sharing),solved
9,"input 0's mram tile must be a whole number of 8-byte DMA granules, i.e. a multiple of 2 elements",solved


Next: how one of these configurations becomes a program, and what it costs (notebook 2).